In [1]:
import os
import re
import html
from pathlib import Path
from collections import Counter

In [2]:
import re
import html
from pathlib import Path

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/preprocessed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MOJIBAKE_MAP = {
    "â€—": "—", "â€™": "'", "â€¢": "•",
    "â˜…": "★", "â˜†": "☆", "â€œ": '"',
    "â€\x9d": '"', "â€": '"', "Ã©": "é",
    "Â£": "£", "Â°": "°", "Â": "",
}

FOOTER_PATTERNS = [
    r"© \d{4} HotelCorp\..*",
    r"© \d{4} TravelInfo Ltd\..*",
    r"© \d{4} Premium Hotels Group.*",
    r"© \d{4} Luxury Hotels International.*",
    r"© \d{2,4}-\d{2,4} StayGuide Inc.*",
    r"This page was last updated on \d{4}-\d{2}-\d{2}\..*",
]


def clean(text: str) -> str:
    for tag in (r"<!--.*?-->", r"<script.*?>.*?</script>",
                r"<footer.*?>.*?</footer>", r"<nav.*?>.*?</nav>",
                r"<div class=['\"].*?>.*?</div>", r"<p class=['\"].*?>.*?</p>"):
        text = re.sub(tag, "", text, flags=re.DOTALL)
    text = re.sub(r"<[^>]+>", "", text)

    text = html.unescape(text)
    for bad, good in sorted(MOJIBAKE_MAP.items(), key=lambda x: -len(x[0])):
        text = text.replace(bad, good)

    for pattern in FOOTER_PATTERNS:
        text = re.sub(pattern, "", text)

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n", "\n\n", text)
    return text.strip()


if __name__ == "__main__":
    raw_files = list(RAW_DIR.glob("*.txt"))
    if not raw_files:
        raise FileNotFoundError(f"No .txt files found in '{RAW_DIR}'. Run generate_dataset.py first.")

    for path in raw_files:
        text = path.read_text(encoding="utf-8")
        (PROCESSED_DIR / path.name).write_text(clean(text), encoding="utf-8")

    print(f"Preprocessed {len(raw_files)} files → '{PROCESSED_DIR}/'")


Preprocessed 39 files → 'data\preprocessed/'


In [4]:
%pip install langchain-groq


  Attempting uninstall: groq

    Found existing installation: groq 1.0.0

    Uninstalling groq-1.0.0:

      Successfully uninstalled groq-1.0.0

   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   -----------------------

In [5]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from pathlib import Path

C:\Users\Juveria\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY  = os.getenv("GROQ_API_KEY", "YOUR_GROQ_API_KEY_HERE")
GROQ_MODEL    = "llama-3.3-70b-versatile"
PROCESSED_DIR = Path("data/preprocessed")
CHUNK_SIZE    = 512
OVERLAP       = 64
TOP_K         = 5

In [7]:
def load_documents(directory: Path) -> list[dict]:
    return [
        {"text": p.read_text(encoding="utf-8"), "source": p.name, "modality": "text"}
        for p in sorted(directory.glob("*.txt"))
    ]

In [8]:
def chunk_documents(documents: list[dict]) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=OVERLAP)
    out = []
    for item in documents:
        for j, split in enumerate(splitter.split_text(item["text"])):
            out.append(Document(
                page_content=split,
                metadata={**{k: v for k, v in item.items() if k != "text"}, "chunk_index": j}
            ))
    return out

In [9]:
def build_vectorstore(chunks: list[Document]) -> FAISS:
    embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    return FAISS.from_documents(chunks, embedder)

In [10]:
PROMPT = PromptTemplate.from_template("""
You are a hotel information assistant. Answer the question based on the context provided below.
If multiple hotels in the context match the query, mention ALL of them.
Do not invent details that are not in the context.
If the context contains no relevant information at all, say "I don't have that information."

Context:
{context}

Question:
{question}

Answer:""")

def format_docs(docs: list) -> str:
    return "\n\n".join(
        f"[Source: {d.metadata.get('source', 'unknown')}]\n{d.page_content}"
        for d in docs
    )

In [11]:
def build_chain(vectorstore: FAISS):
    llm = ChatGroq(model=GROQ_MODEL, api_key=GROQ_API_KEY, temperature=0)
    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": TOP_K, "lambda_mult": 0.7}
    )
    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | PROMPT
        | llm
        | StrOutputParser()
    )
    return chain, retriever

In [12]:
if __name__ == "__main__":
    print("Loading documents...")
    docs = load_documents(PROCESSED_DIR)
    print(f"  {len(docs)} documents loaded")

    print("Chunking...")
    chunks = chunk_documents(docs)
    print(f"  {len(chunks)} chunks created")

    print("Building FAISS index...")
    vectorstore = build_vectorstore(chunks)
    print("  Index ready")

    chain, retriever = build_chain(vectorstore)

    queries = [
        "Which hotels have free WiFi or complimentary breakfast?",
        "What is the cancellation policy of Hotel del Coronado?",
        "Suggest a hotel with excellent reviews near the beach.",
    ]

    for q in queries:
        print(f"\n{'─'*60}")
        print(f"Q: {q}")
        print(f"A: {chain.invoke(q)}")

Loading documents...
  39 documents loaded
Chunking...
  58 chunks created
Building FAISS index...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Index ready

────────────────────────────────────────────────────────────
Q: Which hotels have free WiFi or complimentary breakfast?
A: The Peninsula Hong Kong, The Ritz London, and Hotel del Coronado have complimentary WiFi. 

The Peninsula Hong Kong and The Ritz London have complimentary breakfast options (breakfast buffet at The Lobby and full English breakfast at The Ritz Restaurant, respectively). Hotel del Coronado has a lovely breakfast at Sheerwater, but it's not specified as complimentary.

────────────────────────────────────────────────────────────
Q: What is the cancellation policy of Hotel del Coronado?
A: The cancellation policy of Hotel del Coronado is as follows: 
- Free cancellation up to 72 hours before check-in.
- Cancellations within 72 hours: 1 night penalty.

────────────────────────────────────────────────────────────
Q: Suggest a hotel with excellent reviews near the beach.
A: Based on the context, the following hotels have excellent reviews near the beach: 



In [14]:
# Task 4 – Evaluation

RELEVANT = {
    "Which hotels have free WiFi and complimentary breakfast?":
        {"The_Ritz_London_amenities.txt", "The_Peninsula_Hong_Kong_amenities.txt"},
    "What is the cancellation policy of Hotel del Coronado?":
        {"Hotel_del_Coronado_policy.txt"},
    "Suggest a hotel with excellent reviews near the beach.":
        {"Hotel_del_Coronado_review_1.txt", "OneAndOnly_Royal_Mirage_review_1.txt"},
}

rr_scores = []

for q, relevant in RELEVANT.items():
    docs    = retriever.invoke(q)
    answer  = chain.invoke(q)
    sources = [d.metadata["source"] for d in docs]
    k       = len(docs)

    print(f"\n{'='*60}\nQ: {q}\n")

    print("Retrieved Chunks:")
    for i, doc in enumerate(docs, 1):
        preview = doc.page_content[:180].replace("\n", " ").strip()
        print(f"  [{i}] {doc.metadata['source']}\n      {preview}...")

    print(f"\nAnswer:\n{answer}")

    hits       = [s for s in sources if s in relevant]
    p          = len(hits) / k
    first_rank = next((i for i, s in enumerate(sources, 1) if s in relevant), None)
    rr         = 1 / first_rank if first_rank else 0.0
    rr_scores.append(rr)

    print(f"\nPrecision@{k} = {len(hits)}/{k} = {p:.2f}")
    if first_rank:
        print(f"Reciprocal Rank = 1/{first_rank} = {rr:.2f}")
    else:
        print(f"Reciprocal Rank = 0  (no relevant doc in top-{k})")

n      = len(rr_scores)
mrr    = sum(rr_scores) / n
rr_str = " + ".join(f"{r:.2f}" for r in rr_scores)
print(f"\n{'='*60}")
print(f"MRR = (1/{n}) x ({rr_str}) = {mrr:.3f}")

# Failure case – generation faithfulness failure observed in Q1
print(f"\n{'='*60}\nFailure Case\n")
print("Query: Which hotels have free WiFi and complimentary breakfast?")
print("Retrieved rank 1: The_Peninsula_Hong_Kong_amenities.txt")
print('LLM answer:       "The Ritz London has complimentary WiFi... (included in most rates)"')
print("""
The Peninsula Hong Kong amenities chunk was retrieved at rank 1 and explicitly
states complimentary high-speed WiFi and a breakfast buffet. The LLM ignored it
and only cited Ritz London (rank 2). Retrieval was correct (RR = 1.00, both docs
in top 2), but the generator produced an incomplete answer.

This is a generation faithfulness failure — the "lost in the middle" effect where
LLMs under-use chunks that appear early in a long context when a later chunk already
provides a plausible answer. The fix applied was adding the instruction
"If multiple hotels match, mention ALL of them" to the prompt, which reduced but
did not fully eliminate the issue with this model at temperature=0.""")


Q: Which hotels have free WiFi and complimentary breakfast?

Retrieved Chunks:
  [1] The_Peninsula_Hong_Kong_amenities.txt
      • Complimentary high-speed WiFi throughout the hotel.  • Breakfast buffet at The Lobby (continental + dim sum). • The Peninsula Spa by ESPA: indoor heated pool, jacuzzi, steam room...
  [2] Hotel_del_Coronado_review_1.txt
      Complimentary WiFi was adequate.   Breakfast at Sheerwater was lovely with ocean views "” fresh fruit, avocado toast, and excellent coffee. The hotel's history is fascinating "” th...
  [3] The_Ritz_London_amenities.txt
      • Complimentary WiFi in all rooms and public spaces.  • Full English breakfast served at The Ritz Restaurant (included in most rates). • Afternoon Tea at The Palm Court "” advance...
  [4] Waldorf_Astoria_New_York_amenities.txt
      • WiFi included in destination fee ($50/night).  • Breakfast at Peacock Alley: American classics and pastries. • The Bull & Bear: legendary steakhouse and cocktail bar. • Guerlain...

In [15]:
# Task 5 – Hallucination Control
#
# Technique: Strict context-only prompting
# Without this constraint the LLM blends retrieved context with its training-data
# knowledge, producing specific figures (prices, dates, counts) that sound
# authoritative but cannot be verified against the retrieved chunks.

PERMISSIVE_PROMPT = PromptTemplate.from_template("""
You are a knowledgeable hotel concierge. Answer the question as helpfully as possible.

Context:
{context}

Question:
{question}

Answer:""")

STRICT_PROMPT = PromptTemplate.from_template("""
You are a hotel information assistant. Answer the question using ONLY the context below.
If the answer is not explicitly stated in the context, say "I don't have that information."
Do not add details from outside the context.

Context:
{context}

Question:
{question}

Answer:""")

def run_with_prompt(prompt_template, query):
    llm = ChatGroq(model=GROQ_MODEL, api_key=GROQ_API_KEY, temperature=0)
    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt_template
        | llm
        | StrOutputParser()
    )
    return chain.invoke(query)

# The dataset mentions Afternoon Tea at The Ritz ("advance booking essential")
# but contains no pricing. The LLM has strong training-data knowledge of the price.
probe = "How much does Afternoon Tea at The Ritz London cost per person, and is booking required?"

print("Hallucination Control – Before / After")
print(f"\nQuery: {probe}\n")

print("BEFORE (permissive – no restriction on knowledge source):")
print("-" * 60)
before = run_with_prompt(PERMISSIVE_PROMPT, probe)
print(before)

print("\nAFTER (strict – context only):")
print("-" * 60)
after = run_with_prompt(STRICT_PROMPT, probe)
print(after)

print("""
Why this works:
  The dataset mentions Afternoon Tea exists and requires advance booking, but
  contains no pricing. The permissive prompt allows the LLM to supplement the
  retrieved context with its training-data knowledge of The Ritz's famous Afternoon
  Tea price — producing a confident, specific figure that is unverifiable from the
  retrieved chunks. This is the core hallucination risk in RAG: plausible retrieval
  + unconstrained generation = undetectable fabrication.

  The strict prompt blocks that path. The LLM can confirm booking is required
  (that IS in context) but must decline on price (that is NOT), making the
  boundary between grounded and fabricated information explicit and auditable.
""")

Hallucination Control – Before / After

Query: How much does Afternoon Tea at The Ritz London cost per person, and is booking required?

BEFORE (permissive – no restriction on knowledge source):
------------------------------------------------------------
I'm happy to help you with your query about Afternoon Tea at The Ritz London. Unfortunately, the provided sources do not mention the exact cost of Afternoon Tea per person. However, I can tell you that booking is highly recommended, and according to our policy, afternoon tea reservations must be cancelled 24 hours in advance.

If you'd like to make a reservation or inquire about the pricing, I'd be more than happy to assist you with that. Please let me know, and I'll do my best to provide you with the most up-to-date information.

AFTER (strict – context only):
------------------------------------------------------------
I don't have that information.

Why this works:
  The dataset mentions Afternoon Tea exists and requires advance bo